# Transport Operations MIS & Performance Analytics System
## Notebook 01: Data Cleaning & Preprocessing Pipeline
**Organization**: Raftaar Cargo & Transport Services Pvt. Ltd.  
**Author**: Operations Analytics Department  
**Objective**: Ingest raw shipment, vehicle, driver, and expense records, identify missing values and duplicate entries, standardize text casing (cities and status tags), convert timestamps, and compute delivery delays.

In [1]:
import os
import pandas as pd
import numpy as np
from datetime import datetime

print(f"Environment initialized. Pandas version: {pd.__version__} | NumPy version: {np.__version__}")

Environment initialized. Pandas version: 2.3.3 | NumPy version: 1.26.4


### 1. Ingesting Raw Multi-Source Data
In operational transportation environments, data originates from booking terminals, dispatch registers, and fleet logs. Let us inspect the raw shipments dataset.

In [2]:
raw_shipments_path = '../data/raw/shipments.csv'
df_raw = pd.read_csv(raw_shipments_path)
print(f"Raw shipments shape: {df_raw.shape}")
dups = df_raw.duplicated(subset=['Shipment_ID']).sum()
print(f"Duplicate Shipment_IDs identified: {dups}")
df_raw.head(5)

Raw shipments shape: (34, 11)
Duplicate Shipment_IDs identified: 2


,Shipment_ID,Booking_Date,Origin,Destination,Vehicle_ID,Driver_ID,Expected_Delivery,Actual_Delivery,Status,Freight,Distance_KM
0,SHP-2026-0801,2026-09-28,Delhi,Mumbai,VEH-DL01-1024,DRV-101,2026-09-30,2026-09-30,Delivered,65400,1420
1,SHP-2026-0802,2026-09-28,mumbai,Bengaluru,VEH-MH04-4512,DRV-105,2026-09-30,2026-09-30,delivered,48200,980
2,SHP-2026-0803,2026-09-28,DELHI,Kolkata,VEH-DL01-2048,DRV-102,2026-10-01,2026-10-03,Delayed,72000,1530
3,SHP-2026-0804,2026-09-29,Bengaluru,Chennai,VEH-KA02-3109,DRV-108,2026-09-30,2026-09-30,Delivered,22500,345
4,SHP-2026-0805,2026-09-29,Delhi,Ahmedabad,VEH-GJ01-8840,DRV-114,2026-10-01,2026-10-01,Delivered,46800,940


### 2. Deduplication and Text Standardization
Notice inconsistent city text such as `delhi`, `DELHI`, and `Delhi`, as well as status values like `delivered` vs `Delivered`. We apply standardized mapping dictionaries.

In [3]:
# Remove duplicates
df_clean = df_raw.drop_duplicates(subset=['Shipment_ID']).copy()

city_map = {
    'delhi': 'Delhi', 'DELHI': 'Delhi',
    'mumbai': 'Mumbai', 'MUMBAI': 'Mumbai',
    'bengaluru': 'Bengaluru', 'bangalore': 'Bengaluru',
    'kolkata': 'Kolkata', 'calcutta': 'Kolkata',
    'chennai': 'Chennai', 'madras': 'Chennai',
    'hyderabad': 'Hyderabad', 'ahmedabad': 'Ahmedabad'
}

df_clean['Origin'] = df_clean['Origin'].astype(str).str.strip().apply(lambda x: city_map.get(x.lower(), x.title()))
df_clean['Destination'] = df_clean['Destination'].astype(str).str.strip().apply(lambda x: city_map.get(x.lower(), x.title()))

print(f"Deduplication complete. Remaining records: {len(df_clean)}")
print("Standardized Origin distribution:")
print(df_clean['Origin'].value_counts())

Deduplication complete. Remaining records: 32
Standardized Origin distribution:
Origin
Delhi        11
Mumbai        6
Bengaluru     4
Kolkata       4
Hyderabad     2
Chennai       2
Ahmedabad     1
Pune          1
Jaipur        1
Name: count, dtype: int64


### 3. Date Parsing & Delay Days Calculation
Operational Rule: $\text{Delay Days} = \max(0, \text{Actual Delivery} - \text{Expected Delivery})$. If $\text{Delay Days} > 0$, the shipment is marked as `Delayed`.

In [4]:
for col in ['Booking_Date', 'Expected_Delivery', 'Actual_Delivery']:
    df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')

def get_delay(row):
    if pd.isna(row['Actual_Delivery']) or pd.isna(row['Expected_Delivery']):
        return 0
    return max(0, (row['Actual_Delivery'] - row['Expected_Delivery']).days)

df_clean['Delay_Days'] = df_clean.apply(get_delay, axis=1)
df_clean['Is_On_Time'] = df_clean['Delay_Days'] == 0

# Export cleaned dataset
df_clean.to_csv('../data/processed/cleaned_shipments.csv', index=False)
print("Cleaned dataset exported to ../data/processed/cleaned_shipments.csv")

Cleaned dataset exported to ../data/processed/cleaned_shipments.csv
